# Exploration du corpus français FQuADRetrieval

Ce notebook vérifie le corpus, génère les embeddings et affiche les documents les plus proches d'une requête française. Les données ont été téléchargées depuis Hugging Face dans `data/raw/fquad_retrieval`.

In [13]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
DATASET_DIR = PROJECT_ROOT / 'data' / 'raw' / 'fquad_retrieval'
assert DATASET_DIR.exists(), f'Corpus introuvable : {DATASET_DIR}'
print(f'Projet : {PROJECT_ROOT}')
print(f'Corpus : {DATASET_DIR}')

Projet : /Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel
Corpus : /Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel/data/raw/fquad_retrieval


In [14]:
corpus = pd.concat([
    pd.read_parquet(DATASET_DIR / 'corpus' / 'validation-00000-of-00001.parquet'),
    pd.read_parquet(DATASET_DIR / 'corpus' / 'test-00000-of-00001.parquet'),
], ignore_index=True).drop_duplicates(subset='_id').reset_index(drop=True)
queries = pd.concat([
    pd.read_parquet(DATASET_DIR / 'queries' / 'validation-00000-of-00001.parquet'),
    pd.read_parquet(DATASET_DIR / 'queries' / 'test-00000-of-00001.parquet'),
], ignore_index=True)
print(f'{len(corpus)} documents ; {len(queries)} requêtes')
corpus.head(3)

309 documents ; 500 requêtes


,_id,text,title
0,pégase_23_55,Le patient que Wilhelm Stekel évoque dans son ...,pégase_23_55
1,tension-transitoire-de-rétablissement_17_8,L'évolution temporelle de la tension sur la bo...,tension-transitoire-de-rétablissement_17_8
2,sadi-carnot-(physicien)_12_19,Nicolas Léonard Sadi Carnot est né à Paris au ...,sadi-carnot-(physicien)_12_19


In [15]:
from src.embeddings import EmbeddingEncoder

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
encoder = EmbeddingEncoder(MODEL_NAME)
documents = (corpus['title'].fillna('') + '. ' + corpus['text']).tolist()
corpus_embeddings = encoder.encode(documents, batch_size=32)
print('Dimension des vecteurs :', encoder.dimension)
print('Matrice des embeddings :', corpus_embeddings.shape)

Loading weights: 100%|█| 199/19


Dimension des vecteurs : 384
Matrice des embeddings : (309, 384)


/Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel/src/embeddings/encoder.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return int(self.model.get_sentence_embedding_dimension())


In [16]:
def rechercher(query: str, limite: int = 5) -> pd.DataFrame:
    query_embedding = encoder.encode(query)
    scores = corpus_embeddings @ query_embedding
    resultats = corpus.iloc[scores.argsort()[::-1][:limite]].copy()
    resultats.insert(0, 'score', scores[scores.argsort()[::-1][:limite]])
    return resultats[['score', '_id', 'title', 'text']]

rechercher('Quelle est la capitale de la France ?', limite=5)

,score,_id,title,text
23,0.587340,histoire-de-la-bretagne_18_22,histoire-de-la-bretagne_18_22,"Politiquement, la région est à contre-courant ..."
178,0.501966,pierre-lambert-de-la-motte_2_65,pierre-lambert-de-la-motte_2_65,L'année 1648 est marquée par de graves trouble...
57,0.498813,pierre-lambert-de-la-motte_2_30,pierre-lambert-de-la-motte_2_30,La ville de Caen est l'une des plus religieuse...
159,0.473843,histoire-de-la-bretagne_18_106,histoire-de-la-bretagne_18_106,La Troisième République a des difficultés à s'...
196,0.464980,pierre-lambert-de-la-motte_2_59,pierre-lambert-de-la-motte_2_59,"Phra Narai, le roi du Siam, attend avec impati..."


## Indexation persistante dans Qdrant

Cette étape enregistre les vecteurs et métadonnées dans la collection `fquad_retrieval`. Elle peut être relancée sans créer de doublons.

In [17]:
from src.config import settings
from src.database import VectorStore

store = VectorStore(
    url=settings.qdrant_url,
    api_key=settings.qdrant_api_key,
    collection_name='fquad_retrieval',
)

passages = [
    {
        'passage_id': str(row['_id']),
        'document_id': str(row['_id']),
        'chunk_index': 0,
        'title': row['title'],
        'text': row['text'],
        'category': 'FQuADRetrieval',
        'source': 'Hugging Face / mteb/FQuADRetrieval',
        'language': 'fr',
    }
    for _, row in corpus.iterrows()
]

store.ensure_collection(encoder.dimension)
store.upsert_passages(passages, corpus_embeddings)
print(f'{len(passages)} documents indexés dans Qdrant, collection : {store.collection_name}')

/Users/joelkouraogo/Desktop/Master 2 IBAM ISIE Soutenance 2026/Projet/projet_vectoriel/src/embeddings/encoder.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return int(self.model.get_sentence_embedding_dimension())


309 documents indexés dans Qdrant, collection : fquad_retrieval


In [18]:
def rechercher_dans_qdrant(question: str, limite: int = 5):
    vecteur_requete = encoder.encode(question)
    return store.search(vecteur_requete, limit=limite)

question = 'Quelle est la capitale de la France ?'
resultats_qdrant = rechercher_dans_qdrant(question)

for rang, resultat in enumerate(resultats_qdrant, start=1):
    print(f'{rang}. Score : {resultat.score:.4f}')
    print(f"Titre : {resultat.payload['title']}")
    print(f"Texte : {resultat.payload['text'][:300]}...\n")

1. Score : 0.5873
Titre : histoire-de-la-bretagne_18_22
Texte : Politiquement, la région est à contre-courant du reste de la France. Lors de la victoire du Bloc national aux législatives de 1919, la Bretagne donne 54 % des voix et 60 % des sièges à la gauche. Le premier maire communiste français est élu aux municipales à Douarnenez. Lors de la victoire du Cartel...

2. Score : 0.5020
Titre : pierre-lambert-de-la-motte_2_65
Texte : L'année 1648 est marquée par de graves troubles en France : après la Guerre de Trente Ans, et la Paix de Westphalie, une guerre civile débute pendant la régence du futur Louis XIV. L'opposition des parlements régionaux à la Régence oblige la famille royale à quitter Paris pour Saint-Germain-en-Laye ...

3. Score : 0.4988
Titre : pierre-lambert-de-la-motte_2_30
Texte : La ville de Caen est l'une des plus religieuses de Normandie, siège de ce que les historiens appellent le « milieu mystique normand ». L'ermitage de Caen, fondé par Jean de Bernières, accueille 

## Suite

Modifiez la requête précédente pour contrôler qualitativement la pertinence. Utilisez ensuite `evaluation.ipynb` pour mesurer automatiquement Recall@k et MRR.